In [ ]:
!pip install pyspark
!pip install findspark


In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Create a Spark session
spark = SparkSession.builder\
    .master("local[*]")\
    .appName("Spark_analytics")\
    .getOrCreate()


In [ ]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/sample_data/dataset.csv")
df.show()


+--------------------+--------------------+------------------+--------------------+-------+----------------+----------------+------------+------+--------+------------+--------------+-------+---------------------+
|      transaction_id|         customer_id|     customer_name|               email|country|            city|product_category|product_name| price|quantity|total_amount|payment_method| status|transaction_timestamp|
+--------------------+--------------------+------------------+--------------------+-------+----------------+----------------+------------+------+--------+------------+--------------+-------+---------------------+
|624ef8ad-95cb-499...|22f7a122-487c-45a...|      Judy Vincent|freemancynthia@ex...|     TH|       East Paul|         Fashion|       build| 92.91|       2|      185.82|  Mobile Money|SUCCESS|  2024-02-29 14:58:21|
|976f73bf-2521-4ce...|8a1e941e-5607-401...|      Donald Brown|bradgould@example...|     GD|       Paulhaven|           Books|      center| 19.55|   

In [ ]:
from pyspark.sql.functions import col

In [36]:
from pyspark.sql.functions import avg, sum, count,desc,concat,lit,lpad,year,month,when,min, max,round



In [ ]:
def revenue_by_country():
  result = df.filter(col("status")=="SUCCESS")\
  .groupBy("country")\
  .agg(
      sum("total_amount").alias("total_revenue"),
      count("transaction_id").alias("transaction_count"),
      avg("total_amount").alias("avg_order_value")
  )\
  .orderBy(desc("total_revenue"))\
  .limit(20)
  result.show(20,truncate=False)

  result.coalesce(1).write.mode("overwrite")\
  .option("header","true")\
  .csv("data/analytics/revenue_by_country")

  return result


In [ ]:
result = revenue_by_country()
result.show()

+-------+------------------+-----------------+-----------------+
|country|total_revenue     |transaction_count|avg_order_value  |
+-------+------------------+-----------------+-----------------+
|PW     |1882594.3200000008|2495             |754.5468216432869|
|CV     |1848953.0699999984|2362             |782.7913082133779|
|LB     |1831530.1200000008|2373             |771.8205309734517|
|BR     |1830029.6599999992|2353             |777.7431619209516|
|IQ     |1826132.1999999979|2396             |762.1586811352245|
|LI     |1825121.769999999 |2401             |760.1506747188668|
|PA     |1820705.5700000008|2342             |777.4148462852266|
|BB     |1817518.100000002 |2384             |762.3817533557055|
|SI     |1816136.710000001 |2420             |750.4697148760334|
|JP     |1814448.9800000002|2363             |767.858222598392 |
|SN     |1809907.0699999994|2362             |766.2604022015239|
|IS     |1809239.1000000006|2370             |763.3920253164559|
|AM     |1808531.46000000

In [ ]:
df.columns

['transaction_id',
 'customer_id',
 'customer_name',
 'email',
 'country',
 'city',
 'product_category',
 'product_name',
 'price',
 'quantity',
 'total_amount',
 'payment_method',
 'status',
 'transaction_timestamp']

In [ ]:
def monthly_revenue_trend():
    result = df.filter(col("status") == "SUCCESS") \
        .withColumn("year_month",
                   concat(year("transaction_timestamp"),
                         lit("-"),
                         lpad(month("transaction_timestamp"), 2, "0"))) \
        .groupBy("year_month") \
        .agg(
            sum("total_amount").alias("monthly_revenue"),
            count("transaction_id").alias("transaction_count"),
            avg("total_amount").alias("avg_order_value")
        ) \
        .orderBy("year_month")

    result.show(50)

    # Save
    result.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("data/analytics/monthly_revenue_trend")

    return result

In [ ]:
trend = monthly_revenue_trend()

+----------+--------------------+-----------------+-----------------+
|year_month|     monthly_revenue|transaction_count|  avg_order_value|
+----------+--------------------+-----------------+-----------------+
|   2023-12|  2276902.5899999985|             3018|754.4408846918484|
|   2024-01|       1.413406012E7|            18837|750.3349854010723|
|   2024-02| 1.367035686999999E7|            18129|754.0601726515522|
|   2024-03| 1.445062673000003E7|            19110|756.1814092098393|
|   2024-04|1.3923872350000016E7|            18466|754.0275289721659|
|   2024-05|1.4302171769999988E7|            18913|756.2085216517733|
|   2024-06|1.3856295910000004E7|            18345| 755.317302262197|
|   2024-07|1.4272212019999998E7|            18766| 760.535650644783|
|   2024-08| 1.445380009999999E7|            19081|757.4969917719192|
|   2024-09|1.3598613930000024E7|            18028|754.3051880408267|
|   2024-10|1.3951485579999987E7|            18755|743.8808627032785|
|   2024-11| 1.41028

In [ ]:
def revenue_by_payment():
    result = df.filter(col("status") == "SUCCESS") \
        .groupBy("payment_method") \
        .agg(
            sum("total_amount").alias("total_revenue"),
            count("transaction_id").alias("transaction_count"),
            avg("total_amount").alias("avg_order_value"),
            (sum("total_amount") / df.filter(col("status") == "SUCCESS")
                .agg(sum("total_amount")).collect()[0][0] * 100)
                .alias("revenue_percentage")
        ) \
        .orderBy(desc("total_revenue"))

    result.show()

    result.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("data/analytics/revenue_by_payment")

    return result

+-------+------------------+-----------------+-----------------+
|country|     total_revenue|transaction_count|  avg_order_value|
+-------+------------------+-----------------+-----------------+
|     PW|1882594.3200000008|             2495|754.5468216432869|
|     CV|1848953.0699999984|             2362|782.7913082133779|
|     LB|1831530.1200000008|             2373|771.8205309734517|
|     BR|1830029.6599999992|             2353|777.7431619209516|
|     IQ|1826132.1999999979|             2396|762.1586811352245|
|     LI| 1825121.769999999|             2401|760.1506747188668|
|     PA|1820705.5700000008|             2342|777.4148462852266|
|     BB| 1817518.100000002|             2384|762.3817533557055|
|     SI| 1816136.710000001|             2420|750.4697148760334|
|     JP|1814448.9800000002|             2363| 767.858222598392|
|     SN|1809907.0699999994|             2362|766.2604022015239|
|     IS|1809239.1000000006|             2370|763.3920253164559|
|     AM|1808531.46000000

In [26]:
revenue=revenue_by_payment()

+--------------+-------------------+-----------------+-----------------+------------------+
|payment_method|      total_revenue|transaction_count|  avg_order_value|revenue_percentage|
+--------------+-------------------+-----------------+-----------------+------------------+
|   Credit Card|8.488573912999947E7|           111780|759.4000637860036| 25.11459600578414|
| Bank Transfer| 8.44163228299996E7|           111395|757.8106991337097|24.975712833488593|
|    Debit Card|8.438556488999966E7|           111363| 757.752259637399| 24.96661268021213|
|  Mobile Money|8.430602078999963E7|           111978|  752.88021566736| 24.94307848051476|
+--------------+-------------------+-----------------+-----------------+------------------+



In [ ]:
def revenue_by_category():
    result = df.filter(col("status") == "SUCCESS") \
        .groupBy("product_category") \
        .agg(
            sum("total_amount").alias("total_revenue"),
            count("transaction_id").alias("transactions"),
            sum("quantity").alias("units_sold"),
            avg("price").alias("avg_price"),
            avg("total_amount").alias("avg_order_value")
        ) \
        .orderBy(desc("total_revenue"))

    result.show()

    result.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("data/analytics/revenue_by_category")

    return result

result.show()

+-------+------------------+-----------------+-----------------+
|country|     total_revenue|transaction_count|  avg_order_value|
+-------+------------------+-----------------+-----------------+
|     PW|1882594.3200000008|             2495|754.5468216432869|
|     CV|1848953.0699999984|             2362|782.7913082133779|
|     LB|1831530.1200000008|             2373|771.8205309734517|
|     BR|1830029.6599999992|             2353|777.7431619209516|
|     IQ|1826132.1999999979|             2396|762.1586811352245|
|     LI| 1825121.769999999|             2401|760.1506747188668|
|     PA|1820705.5700000008|             2342|777.4148462852266|
|     BB| 1817518.100000002|             2384|762.3817533557055|
|     SI| 1816136.710000001|             2420|750.4697148760334|
|     JP|1814448.9800000002|             2363| 767.858222598392|
|     SN|1809907.0699999994|             2362|766.2604022015239|
|     IS|1809239.1000000006|             2370|763.3920253164559|
|     AM|1808531.46000000

In [27]:
revenue_category = revenue_by_category()

+----------------+--------------------+------------+----------+------------------+-----------------+
|product_category|       total_revenue|transactions|units_sold|         avg_price|  avg_order_value|
+----------------+--------------------+------------+----------+------------------+-----------------+
|         Fashion|4.8557842170000315E7|       63862|    191842| 252.8106476464873| 760.355801102382|
|       Groceries| 4.850147614999972E7|       64026|    191635|253.07495486208703| 757.527819167209|
|            Home| 4.849150548000007E7|       64255|    192416|252.46998148004042|754.6728733950677|
|          Health| 4.845991044000012E7|       63922|    191587|252.89646569256337|758.1100472450818|
|           Books|  4.81187421599999E7|       63434|    190877|252.19134738468298|758.5638957026184|
|     Electronics| 4.793210531999996E7|       63458|    189697|252.72475007091245|755.3358964984709|
|          Sports| 4.793206591999988E7|       63559|    190247| 251.7385344325736| 754.1349

**PART 3: QUERY GROUP 2 - PRODUCT ANALYSIS (45 minutes)**
Query 2.1: Top 20 Products by Revenue
Business Question: What are our best-selling products by revenue?

In [ ]:
def top_products_by_revenue():
    result = df.filter(col("status") == "SUCCESS") \
        .groupBy("product_name", "product_category") \
        .agg(
            sum("total_amount").alias("total_revenue"),
            sum("quantity").alias("units_sold"),
            count("transaction_id").alias("order_count"),
            avg("price").alias("avg_price")
        ) \
        .orderBy(desc("total_revenue")) \
        .limit(20)

    result.show(20, truncate=False)

    return result

+-------+------------------+-----------------+-----------------+
|country|     total_revenue|transaction_count|  avg_order_value|
+-------+------------------+-----------------+-----------------+
|     PW|1882594.3200000008|             2495|754.5468216432869|
|     CV|1848953.0699999984|             2362|782.7913082133779|
|     LB|1831530.1200000008|             2373|771.8205309734517|
|     BR|1830029.6599999992|             2353|777.7431619209516|
|     IQ|1826132.1999999979|             2396|762.1586811352245|
|     LI| 1825121.769999999|             2401|760.1506747188668|
|     PA|1820705.5700000008|             2342|777.4148462852266|
|     BB| 1817518.100000002|             2384|762.3817533557055|
|     SI| 1816136.710000001|             2420|750.4697148760334|
|     JP|1814448.9800000002|             2363| 767.858222598392|
|     SN|1809907.0699999994|             2362|766.2604022015239|
|     IS|1809239.1000000006|             2370|763.3920253164559|
|     AM|1808531.46000000

In [28]:
top_product = top_products_by_revenue()

+------------+----------------+-----------------+----------+-----------+------------------+
|product_name|product_category|total_revenue    |units_sold|order_count|avg_price         |
+------------+----------------+-----------------+----------+-----------+------------------+
|south       |Sports          |80058.26000000002|292       |91         |268.4971428571428 |
|inside      |Groceries       |79086.74         |303       |100        |258.17089999999996|
|development |Health          |77981.81999999999|282       |92         |268.0811956521739 |
|the         |Groceries       |77701.15         |270       |87         |284.15436781609196|
|left        |Health          |77636.32999999999|277       |87         |263.19666666666666|
|issue       |Sports          |76464.28         |263       |86         |282.8444186046512 |
|throw       |Books           |76423.40000000002|263       |82         |283.3408536585366 |
|quite       |Fashion         |75779.23         |268       |96         |264.7976

In [ ]:
def top_products_by_quantity():
    result = df.filter(col("status") == "SUCCESS") \
        .groupBy("product_name", "product_category") \
        .agg(
            sum("quantity").alias("total_quantity"),
            sum("total_amount").alias("total_revenue"),
            count("transaction_id").alias("order_count"),
            avg("price").alias("avg_price")
        ) \
        .orderBy(desc("total_quantity")) \
        .limit(20)

    result.show(20, truncate=False)

    result.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("data/analytics/top_products_quantity")

    return result

result.show()

+-------+------------------+-----------------+-----------------+
|country|     total_revenue|transaction_count|  avg_order_value|
+-------+------------------+-----------------+-----------------+
|     PW|1882594.3200000008|             2495|754.5468216432869|
|     CV|1848953.0699999984|             2362|782.7913082133779|
|     LB|1831530.1200000008|             2373|771.8205309734517|
|     BR|1830029.6599999992|             2353|777.7431619209516|
|     IQ|1826132.1999999979|             2396|762.1586811352245|
|     LI| 1825121.769999999|             2401|760.1506747188668|
|     PA|1820705.5700000008|             2342|777.4148462852266|
|     BB| 1817518.100000002|             2384|762.3817533557055|
|     SI| 1816136.710000001|             2420|750.4697148760334|
|     JP|1814448.9800000002|             2363| 767.858222598392|
|     SN|1809907.0699999994|             2362|766.2604022015239|
|     IS|1809239.1000000006|             2370|763.3920253164559|
|     AM|1808531.46000000

In [29]:
top_products_by_quantity=top_products_by_quantity()

+------------+----------------+--------------+-----------------+-----------+------------------+
|product_name|product_category|total_quantity|total_revenue    |order_count|avg_price         |
+------------+----------------+--------------+-----------------+-----------+------------------+
|inside      |Groceries       |303           |79086.74         |100        |258.17089999999996|
|carry       |Fashion         |293           |73860.97         |90         |250.61355555555556|
|south       |Sports          |292           |80058.26000000002|91         |268.4971428571428 |
|plan        |Groceries       |291           |74350.71         |88         |247.5803409090909 |
|less        |Fashion         |289           |70662.61999999998|92         |248.65641304347827|
|certainly   |Health          |287           |72249.95999999999|90         |253.59533333333326|
|which       |Health          |284           |67797.4          |89         |249.58404494382026|
|gas         |Sports          |283      

In [33]:
def avg_price_by_category():
    result = df.groupBy("product_category") \
        .agg(
            avg("price").alias("avg_price"),
            min("price").alias("min_price"),
            max("price").alias("max_price"),
            count("transaction_id").alias("product_count")
        ) \
        .orderBy(desc("avg_price"))

    result.show()

    return result

In [34]:
avg_price_by_category=avg_price_by_category()

+----------------+------------------+---------+---------+-------------+
|product_category|         avg_price|min_price|max_price|product_count|
+----------------+------------------+---------+---------+-------------+
|          Health|252.93238772612852|      5.0|   499.99|        71143|
|       Groceries|252.92401073511587|     5.01|   499.98|        70982|
|         Fashion|252.78429810869585|     5.02|    500.0|        71115|
|     Electronics|252.75742221844035|     5.01|    500.0|        70518|
|           Books|252.36052276822952|      5.0|   499.99|        70471|
|            Home|252.26296502986645|     5.01|    500.0|        71318|
|          Sports|251.67398356272807|     5.01|   499.97|        70693|
+----------------+------------------+---------+---------+-------------+



In [37]:
def product_failure_rates():
    result = df.groupBy("product_name", "product_category") \
        .agg(
            count("transaction_id").alias("total_transactions"),
            sum(when(col("status") == "FAILED", 1).otherwise(0)).alias("failed_count"),
            sum(when(col("status") == "SUCCESS", 1).otherwise(0)).alias("success_count")
        ) \
        .withColumn("failure_rate",
                   round((col("failed_count") / col("total_transactions")) * 100, 2)) \
        .filter(col("total_transactions") >= 10) \
        .orderBy(desc("failure_rate")) \
        .limit(20)

    result.show(20, truncate=False)

    return result
product_failure_rates= product_failure_rates()


+------------+----------------+------------------+------------+-------------+------------+
|product_name|product_category|total_transactions|failed_count|success_count|failure_rate|
+------------+----------------+------------------+------------+-------------+------------+
|concern     |Home            |61                |16          |45           |26.23       |
|dinner      |Books           |66                |16          |50           |24.24       |
|difference  |Fashion         |54                |13          |41           |24.07       |
|by          |Sports          |63                |15          |48           |23.81       |
|explain     |Home            |72                |17          |55           |23.61       |
|city        |Books           |73                |17          |56           |23.29       |
|other       |Sports          |74                |17          |57           |22.97       |
|anything    |Groceries       |62                |14          |48           |22.58       |

In [38]:
def top_customers():
    result = df.filter(col("status") == "SUCCESS") \
        .groupBy("customer_id", "customer_name", "country") \
        .agg(
            sum("total_amount").alias("total_spent"),
            count("transaction_id").alias("order_count"),
            avg("total_amount").alias("avg_order_value"),
            sum("quantity").alias("total_items")
        ) \
        .orderBy(desc("total_spent")) \
        .limit(50)

    result.show(50, truncate=False)


    return result

top_customers = top_customers()


+------------------------------------+-------------------------+-------+-----------+-----------+---------------+-----------+
|customer_id                         |customer_name            |country|total_spent|order_count|avg_order_value|total_items|
+------------------------------------+-------------------------+-------+-----------+-----------+---------------+-----------+
|16f71162-862f-45ce-a462-ab4988181d37|Jennifer Hoover          |GH     |2500.0     |1          |2500.0         |5          |
|562f1867-14ef-4f1b-9022-52b7ce356b05|Samuel Butler            |RU     |2499.9     |1          |2499.9         |5          |
|7cef27c2-e8a6-4d3e-b0f6-337c57b15354|Brenda Rose              |FJ     |2499.85    |1          |2499.85        |5          |
|9e2aaefa-e199-4e29-aa03-c6dc89a6ce03|Amanda Bryant            |WS     |2499.85    |1          |2499.85        |5          |
|5dd064e4-b331-471e-9e45-dee0f10320d1|William Hernandez        |GQ     |2499.75    |1          |2499.75        |5          |


In [39]:
def customer_value_distribution():
    customer_ltv = df.filter(col("status") == "SUCCESS") \
        .groupBy("customer_id") \
        .agg(sum("total_amount").alias("lifetime_value"))

    result = customer_ltv.select(
        count("customer_id").alias("total_customers"),
        avg("lifetime_value").alias("avg_ltv"),
        min("lifetime_value").alias("min_ltv"),
        max("lifetime_value").alias("max_ltv")
    )

    # Create value buckets
    buckets = customer_ltv.withColumn(
        "value_bucket",
        when(col("lifetime_value") < 100, "< $100")
        .when(col("lifetime_value") < 500, "$100-$500")
        .when(col("lifetime_value") < 1000, "$500-$1K")
        .when(col("lifetime_value") < 5000, "$1K-$5K")
        .otherwise("> $5K")
    ).groupBy("value_bucket").count().orderBy("value_bucket")

    result.show()
    buckets.show()

    return result
customer_value_distribution = customer_value_distribution()

+---------------+-----------------+-------+-------+
|total_customers|          avg_ltv|min_ltv|max_ltv|
+---------------+-----------------+-------+-------+
|         446516|756.9575281512839|    5.0| 2500.0|
+---------------+-----------------+-------+-------+

+------------+------+
|value_bucket| count|
+------------+------+
|   $100-$500|164755|
|     $1K-$5K|129088|
|    $500-$1K|115771|
|      < $100| 36902|
+------------+------+



In [40]:
def purchase_frequency():
    result = df.groupBy("customer_id") \
        .agg(count("transaction_id").alias("purchase_count")) \
        .groupBy("purchase_count") \
        .agg(count("customer_id").alias("customer_count")) \
        .orderBy("purchase_count") \
        .limit(20)

    result.show()


    return result
purchase_frequency = purchase_frequency()

+--------------+--------------+
|purchase_count|customer_count|
+--------------+--------------+
|             1|        496240|
+--------------+--------------+

